# Study 918 — Creation Halt — the teardown

Signed fund-versus-uncapped-twin spreads around six hardcoded issuance suspensions: placebo-standardised announcement CARs at K = 5/10/20, the suspended-regime HAC drift with a block-bootstrap CI, the resumption fade, a leave-one-event-out jackknife, the ruler-quality split, the resumption-date and cost × borrow sweeps, and the planted/null synthetic control.

Every real number is frozen from `docs/results.md` (fingerprint `696cca301c40`, as-of 2026-06-30). One execution lag: announced at `t`, acted at `t+1`, so every window starts at `t+1` and the announcement-day move is excluded from every statistic.

> 💡 **In plain words:** we are asking whether a fund that cannot print new shares drifts richer than an otherwise identical fund that can.

In [1]:
R = {'asof': '2026-06-30', 'fp': '696cca301c40', 'n_events': 6, 'n_announcement': 5, 'car20': {'UNG-2009': -3.3, 'USO-2020': -6.0, 'VXX-2022': 15.41, 'OIL-2022': -7.03, 'BITO-2021': -0.95}, 'z20': {'UNG-2009': -0.24, 'USO-2020': -1.63, 'VXX-2022': 13.84, 'OIL-2022': -2.55, 'BITO-2021': -0.36}, 'pooled_z5': -0.91, 'pooled_t5': -0.25, 'pos5': 3, 'pooled_z10': 2.0, 'pooled_t10': 0.61, 'pos10': 3, 'pooled_z20': 1.81, 'pooled_t20': 0.6, 'pos20': 1, 'ci20_lo': -1.9, 'ci20_hi': 7.93, 'drift': {'UNG-2009': (-20.12, -0.6, -11.01), 'USO-2020': (-351.48, -2.09, -19.01), 'VXX-2022': (17.72, 0.71, 19.59), 'OIL-2022': (-2.11, -0.25, -2.11), 'GBTC-2024': (4.35, 0.67, 158.56), 'BITO-2021': (-9.09, -0.67, -3.83)}, 'fade_car': {'UNG-2009': -28.29, 'USO-2020': 14.58, 'VXX-2022': -18.48, 'OIL-2022': 0.16, 'GBTC-2024': -2.64, 'BITO-2021': 0.24}, 'fade_z': {'UNG-2009': -5.02, 'USO-2020': 4.25, 'VXX-2022': -16.83, 'OIL-2022': 0.0, 'GBTC-2024': -0.31, 'BITO-2021': 0.42}, 'fade_mean_z': -2.91, 'fade_t': -0.96, 'fade_neg': 3, 'jack_drop_vxx_z': -1.19, 'jack_drop_vxx_t': -2.17, 'ruler_exact_z': 13.84, 'ruler_mismatch_z': -1.19, 'ruler_mismatch_t': -2.17, 'era_early_z': -0.24, 'era_late_z': 2.33, 'era_late_t': 0.6, 'net': {'UNG-2009': -12.22, 'USO-2020': -19.52, 'VXX-2022': 17.8, 'OIL-2022': -3.78, 'GBTC-2024': 126.4, 'BITO-2021': -4.8}, 'net_mean': 17.31, 'net_median': -4.29, 'net_pos': 2, 'net_t': 0.77, 'net_ex_gbtc_mean': -4.51, 'net_ex_gbtc_pos': 1, 'borrow30_mean': -27.26, 'borrow30_median': -16.51, 'borrow10_mean': 5.76, 'blind_days': 60, 'blind': {'UNG-2009': -32.98, 'USO-2020': -6.45, 'VXX-2022': 0.57, 'OIL-2022': -2.14, 'GBTC-2024': 96.62, 'BITO-2021': -8.49}, 'blind_mean': 7.86, 'blind_median': -4.29, 'blind_pos': 2, 'blind_t': 0.43, 'n_looks': 30, 'fw_bar': 0.99833, 'vxx_ann_day': 5.81, 'vxx_in_days': 101, 'vxx_in_bps': 17.72, 'vxx_in_total': 19.59, 'vxx_hac_t': 0.71, 'vxx_sharpe': 1.1, 'vxx_ci_lo': -30.3, 'vxx_ci_hi': 69.1, 'vxx_frac_neg': 0.253, 'vxx_car20': 15.41, 'vxx_pct': 0.999, 'vxx_fade': -18.48, 'vxx_fade_pct': 0.0, 'vxx_nplacebo': 1978, 'vxx_pct_indep': 0.99, 'vxx_fade_pct_indep': 0.0, 'vxx_nplacebo_indep': 99, 'vxx_path': {5: 11.42, 10: 19.1, 20: 15.41, 40: -1.06, 60: 1.79, 101: 17.89}, 'fee': {'UNG-2009': -0.54, 'USO-2020': -0.01, 'VXX-2022': -0.02, 'OIL-2022': 0.01, 'GBTC-2024': 0.79, 'BITO-2021': -0.38}, 'gbtc_in_net_fee': 3.56, 'syn_pl_t': 2.56, 'syn_pl_fire': 5, 'syn_pl_bps': 10.42, 'syn_pl_fade': -4.73, 'syn_half_t': 0.94, 'syn_half_fire': 3, 'syn_nl_t': -0.73, 'syn_nl_fire': 1, 'syn_nl_bps': -0.66, 'syn_nl_fade': 0.02}

## The design

For each event, `x_t = direction · (Δlog F_t − Δlog P_t)` where `F` is the capped fund and `P` an uncapped instrument on the same underlying; `direction = +1` for a creation suspension and `−1` for GBTC's redemption freeze. `x` is a self-financing long-short spread, so it is an excess-of-cash quantity by construction. The check `max |raw − excess-of-cash| = 0.00e+00` against BIL is an **algebraic identity, not a finding** — `(r_f − r_c) − (r_p − r_c) ≡ r_f − r_p` on any input — and is run only to prove the code path never races a raw return against an excess one. The economic assumption behind it (a full cash rebate on the short) is false for a hard-to-borrow name, which is why borrow is charged separately and swept 0–30 %/yr. Legs are total-return (`auto_adjust=True`) except `NG=F`, a **price-only** continuous futures print.

The **ruler** tag matters more than anything else here: *exact* means the twin holds the same object (VIXY/VXX share an index; BTC-USD is what GBTC holds); *curve-mismatched* means it sits elsewhere on a futures curve, so the spread carries roll yield as well as premium.

Three contaminants are measured rather than waved away: the two legs' **expense-ratio difference** is inside `x` before any halt effect exists (`fee_drag_bps`); the regime control **excludes the post-resumption fade**, which otherwise depresses the baseline and flatters the gap; and the `hold='halt'` trade **exits on a date nobody knew at entry**, so a blind fixed-horizon exit is reported beside it.

## 1. Announcement CARs, standardised by each pair's own placebo distribution

In [2]:
for k, (zs, pooled, t, pos) in {
    5:  (None, R['pooled_z5'],  R['pooled_t5'],  R['pos5']),
    10: (None, R['pooled_z10'], R['pooled_t10'], R['pos10']),
    20: (None, R['pooled_z20'], R['pooled_t20'], R['pos20']),
}.items():
    print(f'K={k:>2}: pooled mean z {pooled:+.2f}  cross-event t {t:+.2f}  positive {pos}/5')
print(f"\nK=20 event-resample CI on the pooled mean z: [{R['ci20_lo']:+.2f}, {R['ci20_hi']:+.2f}]")
print()
for key in R['car20']:
    print(f"  {key:<10s} CAR20 {R['car20'][key]:+7.2f}%   z {R['z20'][key]:+7.2f}")

K= 5: pooled mean z -0.91  cross-event t -0.25  positive 3/5
K=10: pooled mean z +2.00  cross-event t +0.61  positive 3/5
K=20: pooled mean z +1.81  cross-event t +0.60  positive 1/5

K=20 event-resample CI on the pooled mean z: [-1.90, +7.93]

  UNG-2009   CAR20   -3.30%   z   -0.24
  USO-2020   CAR20   -6.00%   z   -1.63
  VXX-2022   CAR20  +15.41%   z  +13.84
  OIL-2022   CAR20   -7.03%   z   -2.55
  BITO-2021  CAR20   -0.95%   z   -0.36


The pooled mean flips sign between K = 5 (−0.91) and K = 10/20 (+2.00/+1.81) and the cross-event *t* never exceeds +0.61. With five events and four degrees of freedom this is the honest statistic, and it is a null.

**Two inference caveats that change how the headline reads.** First, `z` is a scale, not a p-value: these spreads are strongly mean-reverting and fat-tailed, so VXX's `z = +13.84` sits at empirical percentile 0.990, not at the 1e−43 a normal table implies. Second, the default placebo pool steps one session at a time, so its ~2,000 windows overlap by 19/20 — VXX's tape holds **99 independent** 20-day windows, not 1,978, and `pct_indep` reports on that pool. The design inspects **30 percentiles** (5 events × 3 horizons × 2 legs), so the family-wise 5% bar for one look is 0.99833 — above the study's single best number.

> 💡 **In plain words:** across the five halts there is no reliable pattern — one of them dominates every number, and even that one is less extreme than the raw denominator suggests.

## 2. Drift while suspended (HAC *t* on the daily signed spread)

In [3]:
print(f"{'event':<11s}{'bps/day':>10s}{'HAC t':>8s}{'total %':>10s}"
      f"{'fee bps':>10s}{'net of fee':>12s}")
for key, (bps, t, tot) in R['drift'].items():
    fee = R['fee'][key]
    print(f'{key:<11s}{bps:>10.2f}{t:>8.2f}{tot:>10.2f}{fee:>10.2f}{bps-fee:>12.2f}')
print('\nthe two exact-ruler events (VXX vs VIXY, GBTC vs spot BTC) are the only positive ones')
print(f"...and {R['fee']['GBTC-2024']/R['drift']['GBTC-2024'][0]:.0%} of GBTC's drift is "
      f"just its 2%/yr fee against an unfeed spot ruler ({R['gbtc_in_net_fee']:+.2f} bps/d net)")

event         bps/day   HAC t   total %   fee bps  net of fee
UNG-2009       -20.12   -0.60    -11.01     -0.54      -19.58
USO-2020      -351.48   -2.09    -19.01     -0.01     -351.47
VXX-2022        17.72    0.71     19.59     -0.02       17.74
OIL-2022        -2.11   -0.25     -2.11      0.01       -2.12
GBTC-2024        4.35    0.67    158.56      0.79        3.56
BITO-2021       -9.09   -0.67     -3.83     -0.38       -8.71

the two exact-ruler events (VXX vs VIXY, GBTC vs spot BTC) are the only positive ones
...and 18% of GBTC's drift is just its 2%/yr fee against an unfeed spot ruler (+3.56 bps/d net)


Note the HAC *t*s: even VXX's +19.6% over the freeze is *t* = +0.71 on a daily basis, and its block-bootstrap CI on the daily mean is [-30.3, +69.1] bps/day with 25.3% of resamples negative. The divergence is a level shift delivered in lumps, not a harvestable daily accrual — which is precisely why the *event-window* standardisation, not the daily *t*, is the right test here.

## 3. The resumption fade (K = 20, window starts resume+1)

In [4]:
for key in R['fade_car']:
    print(f"  {key:<10s} CAR {R['fade_car'][key]:+7.2f}%   z {R['fade_z'][key]:+7.2f}")
print(f"\npooled mean fade z {R['fade_mean_z']:+.2f}  cross-event t {R['fade_t']:+.2f}  "
      f"negative {R['fade_neg']}/6")

  UNG-2009   CAR  -28.29%   z   -5.02
  USO-2020   CAR  +14.58%   z   +4.25
  VXX-2022   CAR  -18.48%   z  -16.83
  OIL-2022   CAR   +0.16%   z   +0.00
  GBTC-2024  CAR   -2.64%   z   -0.31
  BITO-2021  CAR   +0.24%   z   +0.42

pooled mean fade z -2.91  cross-event t -0.96  negative 3/6


UNG (z = −5.02) and VXX (z = −16.83) are the two funds that demonstrably carried a premium, and both dump it. The pool is still *t* = −0.96: three of six events never had a premium to lose.

> 💡 **In plain words:** the collapse-on-restart half of the story holds wherever there was something to collapse.

## 4. Jackknife, ruler split, era cut, date sweep

In [5]:
print(f"drop VXX-2022  -> pooled mean z {R['jack_drop_vxx_z']:+.2f}  t {R['jack_drop_vxx_t']:+.2f}  (sign flips)")
print(f"ruler = exact             (n=1): mean z {R['ruler_exact_z']:+.2f}")
print(f"ruler = curve-mismatched  (n=4): mean z {R['ruler_mismatch_z']:+.2f}  t {R['ruler_mismatch_t']:+.2f}  0/4 positive")
print(f"era cut 2020: early (n=1) z {R['era_early_z']:+.2f} | late (n=4) z {R['era_late_z']:+.2f} (t {R['era_late_t']:+.2f}) -- uninformative by construction")
print('resumption-date sweep +/-10 bd: mean fade z stays in [-5.94, -2.80]; the in-halt drift does NOT survive it')

drop VXX-2022  -> pooled mean z -1.19  t -2.17  (sign flips)
ruler = exact             (n=1): mean z +13.84
ruler = curve-mismatched  (n=4): mean z -1.19  t -2.17  0/4 positive
era cut 2020: early (n=1) z -0.24 | late (n=4) z +2.33 (t +0.60) -- uninformative by construction
resumption-date sweep +/-10 bd: mean fade z stays in [-5.94, -2.80]; the in-halt drift does NOT survive it


The calendar era cut is honestly useless here (1 event before 2020, 4 after) — the hardcoded list is too small and too clustered. The cut that *does* carry information is by ruler quality, and it is damning in a specific way: the four curve-mismatched pairs are collectively **negative** (*t* = −2.17), which is not evidence against the mechanism so much as evidence that a roll-mismatched ruler cannot measure it. Study [661](../../661-uso-roll-decay/) quantifies exactly the confound at work.

The resumption-date sweep is the ASSUMPTION check: half the resumption dates are our public reading rather than a filing date. The *fade* survives ±10 business days; the in-halt drift number does not (USO's window is six sessions long), so we do not report the in-halt drift as a result.

## 5. Tradability — one dollar long the fund, one dollar short the twin

10 bps one-way × NAV on **both** legs at entry and exit (four crossings), a **daily rebalancing charge** of `10 bps × Σ|xₜ|` — because `exp(Σx)` is the return of a continuously dollar-neutral position and holding one flat costs turnover, 5.73% rather than 0.40% over GBTC's 2,183 sessions — and 3%/yr borrow on the short leg per calendar day held.

The `hold='halt'` column exits **on the resumption date**. That is a hindsight exit: at `t+1` nobody knows the halt will run 101 sessions (VXX) or 6 (USO), and for the APPROX events the date is our own reading. `hold='blind'` removes the assumption entirely — enter at `t+1`, exit after a fixed 60 sessions.

In [6]:
print(f"{'event':<11s}{'hindsight':>13s}{'blind 60d':>13s}")
for key in R['net']:
    print(f"{key:<11s}{R['net'][key]:>12.2f}%{R['blind'][key]:>12.2f}%")
print(f"\n{'mean':<11s}{R['net_mean']:>12.2f}%{R['blind_mean']:>12.2f}%")
print(f"{'median':<11s}{R['net_median']:>12.2f}%{R['blind_median']:>12.2f}%")
print(f"{'cross-ev t':<11s}{R['net_t']:>13.2f}{R['blind_t']:>13.2f}")
print(f"\nex-GBTC (an 8.7-year regime, not a trade): mean {R['net_ex_gbtc_mean']:+.2f}%  "
      f"positive {R['net_ex_gbtc_pos']}/5")
print(f"borrow sensitivity: mean net {R['borrow10_mean']:+.2f}% at 10%/yr, "
      f"{R['borrow30_mean']:+.2f}% at 30%/yr")

event          hindsight    blind 60d
UNG-2009         -12.22%      -32.98%
USO-2020         -19.52%       -6.45%
VXX-2022          17.80%        0.57%
OIL-2022          -3.78%       -2.14%
GBTC-2024        126.40%       96.62%
BITO-2021         -4.80%       -8.49%

mean              17.31%        7.86%
median            -4.29%       -4.29%
cross-ev t          0.77         0.43

ex-GBTC (an 8.7-year regime, not a trade): mean -4.51%  positive 1/5
borrow sensitivity: mean net +5.76% at 10%/yr, -27.26% at 30%/yr


**The VXX row is the whole tradability verdict.** Held to the resumption date it nets +17.80%; held blind for 60 sessions it nets +0.57%. The premium round-trips to -1.06% by day 40 and only recovers by the end of the halt, so essentially all of the flagship event's P&L was the *timing of the exit* — which was hindsight. (GBTC's blind number is not a halt trade at all: 60 arbitrary sessions inside an 8.7-year regime.)

Commissions move the answer by ~3 pp across a 0–25 bps grid; borrow moves it by 50 pp across 0–30%/yr. The borrow rate on a capped, squeezed ETP is not published anywhere free, so it is an **ASSUMPTION** and is swept rather than chosen — and between the sweep and the blind exit, the verdict is decided.

## 6. Live synthetic control — the estimator is unbiased and under-powered

**Synthetic, not the real tape.** Six planted pairs per draw, eight seeds each.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from creation_halt import strategy as st
for tag, ss in [('planted', 1.0), ('half', 0.5), ('null', 0.0)]:
    res = [st.synthetic_detect(ss, seed=918 + 11*s) for s in range(8)]
    t = np.array([r['pooled_t'] for r in res])
    b = np.array([r['mean_in_bps'] for r in res])
    f = np.array([r['mean_fade_z'] for r in res])
    print(f'{tag:<8s} pooled t {t.mean():+.2f}  |t|>=2 in {(abs(t)>=2).sum()}/8  '
          f'in-halt {b.mean():+6.2f} bps/d  fade z {f.mean():+.2f}')

planted  pooled t +2.56  |t|>=2 in 5/8  in-halt +10.42 bps/d  fade z -4.73


half     pooled t +0.94  |t|>=2 in 3/8  in-halt  +4.88 bps/d  fade z -2.37


null     pooled t -0.73  |t|>=2 in 1/8  in-halt  -0.66 bps/d  fade z +0.02


The detector recovers a planted 12 bps/day premium (pooled *t* +2.56, fires 5/8), degrades at half strength, and is silent on the null (*t* -0.73, fires 1/8). Two readings follow. First, the harness is unbiased — the real-tape null is a fact about the events, not a broken estimator. Second, **six events buy very little power**: even a genuinely large planted premium clears |*t*| ≥ 2 only 5 times in 8. A pooled null on five real events is therefore weak evidence of absence, which is one more reason the stamp is Mixed rather than None.

## Verdict

- **Signal — Mixed.** In the one suspension with an exact uncapped ruler, VXX repriced **+15.41%** against VIXY over the 20 lagged sessions after the announcement and gave back **-18.48%** after issuance resumed. Stated honestly that is percentile 0.990 and 0.000 of the **99 independent** 20-day windows in the pair's tape — not 1,978 overlapping ones — so against the design's 30 looks (family-wise bar 0.99833) **neither tail clears alone**; the *joint* announce-and-fade pattern, in the two predicted directions on the two named dates, is what survives, and it is n = 1. GBTC, the other exact-ruler case, drifts the predicted way but at HAC *t* = +0.67 with 18% of the drift being nothing but its fee. The pooled result is a null: mean z **+1.81**, cross-event *t* **+0.60**, 1/5 positive at K = 20, sign-flipping across horizons, and *t* = -2.17 without VXX. No |*t*| ≥ 2 on the pooled tape, so not Real; too specific a joint pattern in the one clean pair to call None. **Survivorship:** the event list is hand-curated from *reported* suspensions and excludes TVIX 2012 and the original VXX note whose tapes did not survive delisting — it is biased toward the effect and still cannot pool.
- **Tradability — Mirage.** The flagship winner is an artefact of the exit date: VXX nets **+17.80%** held to the resumption announcement and **+0.57%** on a blind 60-session hold, the only rule available in advance. Median per-event net **-4.29%** at 10 bps / rebalancing / 3%/yr borrow, 2/6 positive, *t* +0.77 (hindsight) and +0.43 (blind); ex-GBTC mean **-4.51%**, 1/5. The decisive input is a borrow rate nobody publishes, and at 30%/yr the mean net is **-27.26%**. Six events, half of them with an assumed resumption date, and no way to tell a VXX from a USO in advance.